# 📡 Customer Churn Prediction — Exploratory Analysis

> End-to-end notebook: EDA → Preprocessing → Model Training → Evaluation

---

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import sys, os
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 30)
print("Libraries loaded ✓")

## 1. Load Data

In [ ]:
df = pd.read_csv("../data/data.csv")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
df.info()

In [ ]:
print("Missing values:")
print(df.isnull().sum()[df.isnull().sum() > 0])

## 2. Exploratory Data Analysis

In [ ]:
# Churn distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
churn_counts = df["Churn"].value_counts()
colors = ["#2ecc71", "#e74c3c"]
axes[0].bar(churn_counts.index, churn_counts.values, color=colors, edgecolor="white")
axes[0].set_title("Churn Distribution", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Count")
for i, (idx, val) in enumerate(zip(churn_counts.index, churn_counts.values)):
    axes[0].text(i, val + 30, f"{val} ({val/len(df)*100:.1f}%)", ha="center", fontweight="bold")

# Pie chart
axes[1].pie(churn_counts.values, labels=["Not Churn", "Churn"],
            colors=colors, autopct="%1.1f%%", startangle=90,
            wedgeprops={"edgecolor": "white", "linewidth": 2})
axes[1].set_title("Churn Proportion", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
# Contract type vs Churn
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col in zip(axes, ["Contract", "InternetService", "PaymentMethod"]):
    ct = pd.crosstab(df[col], df["Churn"], normalize="index") * 100
    ct.plot(kind="bar", ax=ax, color=["#2ecc71", "#e74c3c"], edgecolor="white")
    ax.set_title(f"Churn by {col}", fontweight="bold")
    ax.set_ylabel("Percentage (%)")
    ax.set_xlabel("")
    ax.legend(["Not Churn", "Churn"], loc="upper right")
    ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Numeric distributions by churn
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df_temp = df.copy()
df_temp["TotalCharges"] = pd.to_numeric(df_temp["TotalCharges"], errors="coerce")

for ax, col in zip(axes, ["tenure", "MonthlyCharges", "TotalCharges"]):
    for label, color in [("No", "#2ecc71"), ("Yes", "#e74c3c")]:
        subset = df_temp[df_temp["Churn"] == label][col].dropna()
        ax.hist(subset, bins=30, alpha=0.6, color=color,
                label=f"Churn={label}", edgecolor="white")
    ax.set_title(f"{col} Distribution", fontweight="bold")
    ax.set_xlabel(col)
    ax.legend()

plt.tight_layout()
plt.show()

## 3. Preprocessing

In [ ]:
from src.preprocessing import load_data, preprocess

raw = load_data("../data/data.csv")
X, y, scaler = preprocess(raw, fit_scaler=True)
print(f"
Feature matrix: {X.shape}")
print(f"Churn rate: {y.mean()*100:.1f}%")
X.head(2)

## 4. Model Training & Evaluation

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train: {X_train.shape[0]}  |  Test: {X_test.shape[0]}")

In [ ]:
# Train both models
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")
rf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight="balanced")

lr.fit(X_train, y_train)
rf.fit(X_train, y_train)

results = {
    "Logistic Regression": accuracy_score(y_test, lr.predict(X_test)),
    "Random Forest":       accuracy_score(y_test, rf.predict(X_test)),
}

for name, acc in results.items():
    print(f"{name:25s}  {acc*100:.2f}%")

In [ ]:
# Classification report for Random Forest
print("Classification Report — Random Forest")
print("="*55)
print(classification_report(y_test, rf.predict(X_test), target_names=["Not Churn", "Churn"]))

In [ ]:
# Confusion matrices side by side
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, model, name in zip(axes, [lr, rf], ["Logistic Regression", "Random Forest"]):
    cm = confusion_matrix(y_test, model.predict(X_test))
    ConfusionMatrixDisplay(cm, display_labels=["Not Churn", "Churn"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Confusion Matrix — {name}", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Model comparison bar chart
fig, ax = plt.subplots(figsize=(6, 4))
names = list(results.keys())
accs  = [v*100 for v in results.values()]
bars = ax.bar(names, accs, color=["#3498db", "#9b59b6"], edgecolor="white")
for bar, acc in zip(bars, accs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{acc:.2f}%", ha="center", fontweight="bold")
ax.set_ylim(0, 100)
ax.set_ylabel("Accuracy (%)")
ax.set_title("Model Comparison", fontweight="bold")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 5. Feature Importance (Random Forest)

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).nlargest(15).sort_values()

fig, ax = plt.subplots(figsize=(8, 6))
colors = plt.cm.RdYlGn(np.linspace(0.25, 0.85, len(importances)))
importances.plot(kind="barh", ax=ax, color=colors, edgecolor="white")
ax.set_title("Top 15 Feature Importances (Random Forest)", fontsize=13, fontweight="bold")
ax.set_xlabel("Importance Score")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout()
plt.show()

## 6. Sample Prediction

In [ ]:
import sys
sys.path.insert(0, "..")
from src.predict import predict_churn

customer = {
    "gender": "Male", "SeniorCitizen": 0, "Partner": "Yes",
    "Dependents": "No", "tenure": 12, "PhoneService": "Yes",
    "MultipleLines": "No", "InternetService": "Fiber optic",
    "OnlineSecurity": "No", "OnlineBackup": "No",
    "DeviceProtection": "No", "TechSupport": "No",
    "StreamingTV": "Yes", "StreamingMovies": "Yes",
    "Contract": "Month-to-month", "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 85.0, "TotalCharges": 1020.0,
}

result = predict_churn(customer)
print(f"Prediction  : {result['label']}")
print(f"Probability : {result['probability']*100:.1f}% churn risk")